# Final Project Submission: Pneumonia Detection from Chest X-Rays

## Summary

This notebook is the final project submission for binary pneumonia detection from chest X-ray images. It extends the checkpoint-2 baseline with transfer learning, validation-based model comparison, validation-based threshold selection, selected-model test evaluation, and Grad-CAM explainability.

It documents and generates:
- motivation, problem understanding, and brief related-work context
- dataset source and Kaggle download setup
- original Kaggle split counts and the reason for replacing the tiny validation split
- a reproducible stratified 80/10/10 train/validation/test split
- class imbalance analysis, sample image review, visual observations, and image-quality review examples
- patient-level leakage discussion and the limitation of image-level splitting
- preprocessing and online, training-only augmentation steps
- augmentation count and class-distribution behavior
- medically motivated augmentation discussion, including horizontal flip caveats
- a class-weighted baseline CNN with an architecture table
- baseline training curves, ROC-AUC, PR-AUC, validation-based threshold analysis, classification reports, and confusion matrices
- transfer-learning models using MobileNetV2 and DenseNet121 with ImageNet preprocessing
- validation-based model comparison for Baseline CNN, MobileNetV2, and DenseNet121
- validation-based threshold selection before final selected-model test evaluation
- Grad-CAM examples and a short interpretation summary

Binary image classification - **NORMAL** vs **PNEUMONIA** chest X-rays.

**Dataset:** [Kaggle - Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) by Paul Mooney

**Python version used for this project:** `3.13.8`  
**Dependency file:** `requirements.txt`

**Sections:**
0. Dataset download and setup
0.1. Configuration
0.2. Motivation and related work
1. Data exploration, quality review, and split
2. Preprocessing and augmentation
3. Baseline CNN model
4. Baseline training
5. Baseline evaluation and threshold diagnostics
6. Baseline findings and limitations
7. Transfer learning: MobileNetV2
8. Transfer learning: DenseNet121
9. Evaluation helper for fair model comparison
10. Validation-based model comparison and threshold selection
11. Selected-model test evaluation
12. Grad-CAM explainability
13. Final notes

## How to Run This Notebook

### Local setup in VS Code
Run these commands once in a PowerShell terminal from the project folder:
```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -U pip
python -m pip install -r requirements.txt
```
Then select `.venv` as the notebook kernel in VS Code. After that, you can skip the dependency install cell unless packages are missing or changed.

For Kaggle locally, either set `KAGGLE_API_TOKEN` before launching VS Code/Jupyter, or put your token in `<your-home-folder>\.kaggle\access_token` with no `.txt` extension. On Windows this is usually `C:\Users\<your-username>\.kaggle\access_token`.

To set the environment variable for the current PowerShell session only:
```powershell
$env:KAGGLE_API_TOKEN = "paste-your-token-here"
```

To save it permanently for your Windows user account, run this once, then restart VS Code so the notebook kernel inherits it:
```powershell
[Environment]::SetEnvironmentVariable("KAGGLE_API_TOKEN", "paste-your-token-here", "User")
```

### Google Colab setup
Run the dependency install cell each time you start a fresh Colab runtime. For faster training, go to **Runtime -> Change runtime type -> T4 GPU** before running the training cells.

For Kaggle in Colab, the recommended option is Colab Secrets:
1. Open the left sidebar and click the key icon (**Secrets**).
2. Add a secret named `KAGGLE_API_TOKEN`.
3. Paste your Kaggle token as the value and allow notebook access.

Fallback option: save your token text in Google Drive at `MyDrive/secrets/kaggle_token.txt`. The dataset setup cell can mount Drive and load it if no Colab Secret is found.

### Shared cells
The dependency and dataset setup cells are written to work in both local VS Code and Google Colab. They follow the active notebook kernel, so local runs use the project folder when that kernel is selected, while Colab runs use `/content`.


In [ ]:
import subprocess
import sys
from pathlib import Path

# Install the exact libraries needed by this notebook runtime.
# In VS Code this uses the selected kernel; in Colab it uses the active Colab runtime.
requirements_path = Path('requirements.txt')

if requirements_path.exists():
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)],
        check=True,
    )
else:
    DEPENDENCIES = [
        'tensorflow',
        'numpy',
        'pandas',
        'matplotlib',
        'scikit-learn',
        'seaborn',
        'pillow',
        'kaggle',
    ]
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *DEPENDENCIES],
        check=True,
    )

print(f'Dependencies installed for: {sys.executable}')
print(f'Python version: {sys.version.split()[0]}')


## 0 - Dataset Setup

This section downloads the Chest X-Ray Pneumonia dataset with the Kaggle API. Use the credential setup from **How to Run This Notebook** above before running the cell.

### Local VS Code
- The cell accepts either `KAGGLE_API_TOKEN` or `<your-home-folder>\.kaggle\access_token`.
- If you set `KAGGLE_API_TOKEN`, restart VS Code/Jupyter before running the notebook so the kernel can see it.

### Google Colab
- First choice: add `KAGGLE_API_TOKEN` in the Colab Secrets panel and allow notebook access.
- Fallback: save the token in Drive at `MyDrive/secrets/kaggle_token.txt`.
- The next cell tries Colab Secrets first, then mounts Drive only if the secret is not found.

The notebook expects the dataset at `chest_xray/train...`, `chest_xray/val...`, and `chest_xray/test...`. After download it removes Kaggle archive clutter like `chest_xray/chest_xray` and `chest_xray/__MACOSX`, then prints image counts.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

KAGGLE_DATASET = 'paultimothymooney/chest-xray-pneumonia'
KAGGLE_TOKEN_ENV = 'KAGGLE_API_TOKEN'
COLAB_TOKEN_PATH = Path('/content/drive/MyDrive/secrets/kaggle_token.txt')
KAGGLE_DIR = Path.home() / '.kaggle'
EXPECTED_SPLITS = ('train', 'val', 'test')
EXPECTED_CLASSES = ('NORMAL', 'PNEUMONIA')
VALID_EXTENSIONS = {'.jpeg', '.jpg', '.png', '.bmp'}

try:
    from google.colab import drive, userdata
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

DOWNLOAD_DIR = Path('/content') if RUNNING_IN_COLAB else Path.cwd()
DATASET_DIR = DOWNLOAD_DIR / 'chest_xray'

if RUNNING_IN_COLAB:
    colab_secret = userdata.get(KAGGLE_TOKEN_ENV)
    if colab_secret:
        os.environ[KAGGLE_TOKEN_ENV] = colab_secret
        auth_source = f'Colab Secret: {KAGGLE_TOKEN_ENV}'
    else:
        drive.mount('/content/drive')
        if not COLAB_TOKEN_PATH.exists():
            raise FileNotFoundError(
                f'Kaggle token not found. Add Colab Secret {KAGGLE_TOKEN_ENV}, '
                f'or create {COLAB_TOKEN_PATH}.'
            )
        os.environ[KAGGLE_TOKEN_ENV] = COLAB_TOKEN_PATH.read_text(encoding='utf-8').strip()
        auth_source = f'Colab/Drive token: {COLAB_TOKEN_PATH}'
elif os.environ.get(KAGGLE_TOKEN_ENV):
    auth_source = f'environment variable: {KAGGLE_TOKEN_ENV}'
elif (KAGGLE_DIR / 'access_token').exists():
    token_path = KAGGLE_DIR / 'access_token'
    os.environ[KAGGLE_TOKEN_ENV] = token_path.read_text(encoding='utf-8').strip()
    auth_source = f'token file: {token_path}'
else:
    raise EnvironmentError(
        f'No Kaggle credentials found. Set {KAGGLE_TOKEN_ENV}, '
        f'or create {KAGGLE_DIR / "access_token"}.'
    )

active_token = os.environ.get(KAGGLE_TOKEN_ENV, '')
token_preview = f'{active_token[:4]}...{active_token[-4:]}'
print(f'Kaggle auth source: {auth_source}')
print(f'Kaggle token preview: {token_preview}')

from kaggle.api.kaggle_api_extended import KaggleApi

print(f'Downloading/extracting {KAGGLE_DATASET} into {DOWNLOAD_DIR}')
api = KaggleApi()
api.authenticate()
api.dataset_download_files(KAGGLE_DATASET, path=str(DOWNLOAD_DIR), unzip=True, quiet=False)

missing = [DATASET_DIR / split / cls
           for split in EXPECTED_SPLITS for cls in EXPECTED_CLASSES
           if not (DATASET_DIR / split / cls).is_dir()]
if missing:
    raise FileNotFoundError('Missing expected dataset folders: ' + ', '.join(str(path) for path in missing))

for extra_path in (DATASET_DIR / 'chest_xray', DATASET_DIR / '__MACOSX'):
    if extra_path.exists():
        shutil.rmtree(extra_path)
        print(f'Removed archive clutter: {extra_path}')

print(f'Dataset root: {DATASET_DIR}')
for split in EXPECTED_SPLITS:
    counts = {}
    for cls in EXPECTED_CLASSES:
        folder = DATASET_DIR / split / cls
        counts[cls] = sum(1 for file in folder.iterdir() if file.suffix.lower() in VALID_EXTENSIONS)
    total = sum(counts.values())
    print(f'{split:5s} total={total:4d} | NORMAL={counts["NORMAL"]:4d} | PNEUMONIA={counts["PNEUMONIA"]:4d}')

## 0.1 - Configuration

All paths and hyperparameters are defined here. The raw Kaggle folders are kept unchanged, but the training pipeline below creates a reproducible stratified 80/10/10 split across all downloaded images.

In [ ]:
import sys
from pathlib import Path

# Use /content in Colab and the project folder locally, so the same notebook runs in both places.
BASE_DIR = Path('/content') if 'google.colab' in sys.modules else Path.cwd()
DATASET_DIR = BASE_DIR / 'chest_xray'
OUTPUT_DIR  = BASE_DIR / 'outputs'
FIGURES_DIR = OUTPUT_DIR / 'figures'
RESULTS_DIR = OUTPUT_DIR / 'results'
MODELS_DIR  = OUTPUT_DIR / 'models'

# Expected Kaggle folder names and allowed image formats.
EXPECTED_SPLITS  = ('train', 'val', 'test')
EXPECTED_CLASSES = ('NORMAL', 'PNEUMONIA')
VALID_EXTENSIONS = {'.jpeg', '.jpg', '.png', '.bmp'}

# The original Kaggle validation split is too small, so we create our own split later.
WORKING_SPLIT_RATIOS = {'train': 0.80, 'val': 0.10, 'test': 0.10}

# Core model/training settings kept in one place for easy experiments.
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 10
RANDOM_SEED = 42

# Create output folders before any plots, CSVs, or models are saved.
for d in (FIGURES_DIR, RESULTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  Dataset : {DATASET_DIR}')
print(f'  Outputs : {OUTPUT_DIR}')
print(f'  Working split : {WORKING_SPLIT_RATIOS}')
print(f'  Image size={IMAGE_SIZE}  Batch size={BATCH_SIZE}  Epochs={EPOCHS}')


## 0.2 - Motivation, Related Work, and Ethical Context

Pneumonia is a clinically important lung infection, and chest X-rays are commonly used as part of the diagnostic workflow. A computer-vision model for pneumonia screening is relevant because it can support prioritization and second-opinion workflows, especially when many images must be reviewed. The model in this project is not intended to replace medical professionals; it is a controlled course project for studying how deep learning can be applied to a medical image-classification task.

Convolutional neural networks are a standard approach for image classification because convolutional layers can learn local visual patterns such as edges, textures, and spatial structures. In medical imaging, transfer learning is often used when domain-specific datasets are limited: a model pretrained on a large dataset such as ImageNet can provide reusable low-level visual features, while a smaller task-specific classification head is trained for the medical task.

Related work and benchmark context:
- Wang et al. introduced ChestX-ray14, a large public chest X-ray dataset with 112,120 frontal X-ray images and 14 thoracic disease labels. This helped establish deep learning as a common approach for chest X-ray classification: https://arxiv.org/abs/1705.02315
- Rajpurkar et al. proposed CheXNet, a DenseNet-121 model for pneumonia detection on chest X-rays. This is directly relevant to our DenseNet121 transfer-learning experiment: https://arxiv.org/abs/1711.05225
- Kermany et al. published a pediatric chest X-ray pneumonia dataset and showed that deep learning can identify medical diagnoses from retinal OCT and chest X-ray images. The Kaggle dataset used in this project is based on this data source: https://www.cell.com/cell/fulltext/S0092-8674(18)30154-5

This project compares three approaches:
- a custom CNN baseline trained from scratch,
- MobileNetV2 as a lightweight transfer-learning model,
- DenseNet121 as a larger transfer-learning model with dense feature reuse.

The comparison focuses not only on accuracy, but also on recall, false negatives, ROC-AUC, PR-AUC, and threshold behavior because pneumonia screening is sensitive to missed positive cases.

Ethical and dataset-bias considerations are important in medical AI. This dataset may not represent all patient populations, imaging devices, hospitals, or acquisition protocols. The split is image-level because reliable patient IDs are unavailable, so patient-level leakage cannot be fully excluded. Results should therefore not be interpreted as clinical validation. A deployable medical system would require external validation, patient-level splitting, subgroup analysis, calibration, and expert clinical review.


## 1 - Data Exploration, Quality Review, and Split

This section first checks the raw Kaggle dataset and then creates the working split used by the rest of the notebook.

The original Kaggle validation folder contains only 16 images, which is too small for reliable model selection. With such a small validation set, a few individual images can strongly change validation accuracy or loss.

To follow the recommended project setup, this notebook combines all downloaded images and creates a reproducible stratified 80/10/10 split. Stratification keeps the class proportions similar in train, validation, and test sets. The original Kaggle test folder is therefore treated as part of the raw downloaded data, while the generated 10% test split is held back for selected-model evaluation.

This split gives a much larger validation set while keeping a separate held-out test set for the final selected configuration.

**Patient-level leakage note:** this Kaggle release does not provide explicit patient IDs in a separate metadata table, and the filenames are not documented as reliable patient identifiers. The split below is therefore an image-level stratified split, not a guaranteed patient-level split. If patient IDs were available, the safer approach would be a grouped split such as `GroupShuffleSplit`, ensuring all images from one patient stay in only one of train, validation, or test. This remains an important limitation for medical AI evaluation.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split

def _list_images(directory):
    return sorted(
        f for f in directory.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
    )

# Verify raw Kaggle dataset structure
if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset not found at {DATASET_DIR}. Run the download cell first.')

missing = [DATASET_DIR / split / cls
           for split in EXPECTED_SPLITS for cls in EXPECTED_CLASSES
           if not (DATASET_DIR / split / cls).exists()]
if missing:
    raise FileNotFoundError('Missing expected dataset folders: ' + ', '.join(str(path) for path in missing))

# Count original Kaggle split, then build one manifest containing all images.
rows = []
for original_split in EXPECTED_SPLITS:
    for cls in EXPECTED_CLASSES:
        for image_path in _list_images(DATASET_DIR / original_split / cls):
            rows.append({
                'path': str(image_path),
                'class': cls,
                'original_split': original_split,
            })

all_images_df = pd.DataFrame(rows)
original_summary_df = (
    all_images_df.groupby(['original_split', 'class'])
    .size()
    .reset_index(name='count')
)
original_summary_df.to_csv(RESULTS_DIR / 'dataset_original_kaggle_split.csv', index=False)
print('Original Kaggle split:')
print(original_summary_df.pivot(index='original_split', columns='class', values='count').fillna(0).astype(int))

# Create a reproducible stratified 80/10/10 split over all downloaded images.
train_df, temp_df = train_test_split(
    all_images_df,
    train_size=WORKING_SPLIT_RATIOS['train'],
    stratify=all_images_df['class'],
    random_state=RANDOM_SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['class'],
    random_state=RANDOM_SEED,
)

train_df = train_df.assign(split='train')
val_df = val_df.assign(split='val')
test_df = test_df.assign(split='test')
split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
split_df = split_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
split_df.to_csv(RESULTS_DIR / 'dataset_split_80_10_10.csv', index=False)

summary_df = split_df.groupby(['split', 'class']).size().reset_index(name='count')
summary_df.to_csv(RESULTS_DIR / 'dataset_summary.csv', index=False)
print('\nWorking 80/10/10 split used for training:')
print(summary_df.pivot(index='split', columns='class', values='count').fillna(0).astype(int))

# Class distribution bar plot for the working split.
plt.figure(figsize=(8, 5))
sns.barplot(data=summary_df, x='split', y='count', hue='class')
plt.title('Class Distribution by Working 80/10/10 Split')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150)
plt.show()

# Sample image grid from the working training split.
MAX_PER_CLASS = 4
fig, axes = plt.subplots(2, MAX_PER_CLASS, figsize=(3 * MAX_PER_CLASS, 6), squeeze=False)
for row_idx, cls in enumerate(EXPECTED_CLASSES):
    samples = split_df[(split_df['split'] == 'train') & (split_df['class'] == cls)]['path'].head(MAX_PER_CLASS)
    for col_idx in range(MAX_PER_CLASS):
        ax = axes[row_idx][col_idx]
        if col_idx < len(samples):
            ax.imshow(Image.open(samples.iloc[col_idx]).convert('L'), cmap='gray')
            ax.set_title(cls, fontsize=9)
        ax.axis('off')
plt.suptitle('Sample Training Images', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'example_images_grid.png', dpi=150)
plt.show()

In [ ]:
# Basic image-quality review on a small sample from each class.
# This complements manual visual inspection of the saved example grid.
import numpy as np
from IPython.display import display

QUALITY_SAMPLE_PER_CLASS = 12
quality_rows = []

for cls in EXPECTED_CLASSES:
    class_train_df = split_df[(split_df['split'] == 'train') & (split_df['class'] == cls)]
    sample_paths = class_train_df.sample(
        n=min(QUALITY_SAMPLE_PER_CLASS, len(class_train_df)),
        random_state=RANDOM_SEED,
    )['path']

    for image_path in sample_paths:
        image = Image.open(image_path).convert('L')
        array = np.asarray(image, dtype=np.float32)
        grad_x = np.diff(array, axis=1)
        grad_y = np.diff(array, axis=0)
        sharpness_proxy = float(grad_x.var() + grad_y.var())

        quality_rows.append({
            'path': image_path,
            'class': cls,
            'width': image.width,
            'height': image.height,
            'mean_brightness': float(array.mean()),
            'contrast_std': float(array.std()),
            'sharpness_proxy': sharpness_proxy,
            'low_contrast_flag': bool(array.std() < 25),
            'possible_blur_flag': bool(sharpness_proxy < 50),
        })

quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(RESULTS_DIR / 'image_quality_sample.csv', index=False)

quality_summary_df = (
    quality_df.groupby('class')
    .agg(
        sampled_images=('path', 'count'),
        median_width=('width', 'median'),
        median_height=('height', 'median'),
        median_contrast_std=('contrast_std', 'median'),
        median_sharpness_proxy=('sharpness_proxy', 'median'),
        low_contrast_flags=('low_contrast_flag', 'sum'),
        possible_blur_flags=('possible_blur_flag', 'sum'),
    )
    .reset_index()
)
quality_summary_df.to_csv(RESULTS_DIR / 'image_quality_summary.csv', index=False)

print('Sample image-quality summary:')
display(quality_summary_df)

# Implemented visual inspection support: save the sampled images that look most suspicious
# according to simple contrast and sharpness proxies.
lowest_contrast = quality_df.nsmallest(3, 'contrast_std').assign(review_reason='lowest contrast')
lowest_sharpness = quality_df.nsmallest(3, 'sharpness_proxy').assign(review_reason='lowest sharpness')
review_examples_df = pd.concat([lowest_contrast, lowest_sharpness], ignore_index=True)
review_examples_df.to_csv(RESULTS_DIR / 'image_quality_review_examples.csv', index=False)

fig, axes = plt.subplots(2, 3, figsize=(12, 7), squeeze=False)
for ax, (_, row) in zip(axes.ravel(), review_examples_df.iterrows()):
    image = Image.open(row['path']).convert('L')
    ax.imshow(image, cmap='gray')
    ax.set_title(
        f"{row['class']} - {row['review_reason']}\n"
        f"contrast={row['contrast_std']:.1f}, sharpness={row['sharpness_proxy']:.1f}",
        fontsize=9,
    )
    ax.axis('off')

plt.suptitle('Image Quality Review: Lowest Contrast and Lowest Sharpness Samples', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'image_quality_review_examples.png', dpi=150)
plt.show()

print(f"Saved sample-level quality review to {RESULTS_DIR / 'image_quality_sample.csv'}")
print(f"Saved visual quality examples to {FIGURES_DIR / 'image_quality_review_examples.png'}")


### Manual Image Quality and Visual Inspection Notes

The sample grid and the quality-summary cell above are used as a lightweight manual inspection step. The images are generally recognizable chest X-rays, but the dataset shows expected real-world variation: different image sizes, brightness, contrast, patient positioning, and visible lung-field coverage. Some images appear visually harder than others because pneumonia patterns can be subtle, local, or low contrast.

From a non-expert medical perspective, the visual difference between **NORMAL** and **PNEUMONIA** examples is not always obvious. Pneumonia images may show cloudier or more opaque regions in parts of the lungs, while normal examples often look clearer, but these patterns are not consistently easy to identify without radiology training. This supports the need for quantitative evaluation beyond visual inspection.

The quality checks are not a diagnostic tool. They only help identify whether the image sample contains obvious technical problems such as very low contrast or likely blur. A final medical project would ideally include a larger radiologist-guided quality review.


## 2 - Preprocessing & Data Augmentation

Each image path from the 80/10/10 split manifest is loaded, decoded, resized to **224x224**, converted to three channels, and scaled to `[0, 1]`.

The original X-rays are grayscale, but the model input is converted to three channels so the same preprocessing style can later support transfer learning with pretrained CNNs. Data augmentation is applied only to the training split, never to validation or test data, so evaluation remains deterministic.

**How many training examples are seen after augmentation?** The training split contains **4,684 original images**: **1,266 NORMAL** and **3,418 PNEUMONIA**. Augmentation is performed **online during training** with Keras random layers inside the `tf.data` pipeline. This means the notebook does not create extra augmented files on disk and does not permanently increase the dataset size. Each epoch still iterates over 4,684 training images, but the model can see a different randomly transformed version of an image in different epochs. Stored augmented images on disk: **0**.

**Class distribution after augmentation:** because augmentation is online and is applied uniformly to training batches, it preserves the original training-set class distribution. It does not oversample NORMAL images and does not rebalance the dataset by itself. The baseline training step implements **class weights** to reduce the effect of this imbalance without changing the number of images or duplicating files.

**Medical justification of augmentations:**
- Small rotations model minor patient-positioning differences.
- Small zoom changes model acquisition/cropping variation.
- Mild contrast changes model scanner and exposure differences.
- Horizontal flipping is included cautiously: for this binary task, pneumonia can occur in either lung, and the target label is presence/absence of pneumonia rather than left/right localization. Horizontal flipping would be less appropriate for laterality-sensitive tasks, such as detecting a left-sided condition or estimating anatomical orientation.

The augmentation policy is therefore mild and explicitly limited. It is intended to improve robustness without claiming that all medical image transformations are label-preserving in every clinical task.


In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE
CLASS_TO_LABEL = {cls: index for index, cls in enumerate(EXPECTED_CLASSES)}

def decode_image(path, label):
    # Load each image from disk, force one grayscale channel, and resize consistently.
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=1, expand_animations=False)
    image.set_shape([None, None, 1])
    image = tf.image.resize(image, IMAGE_SIZE)

    # Scale pixels to [0, 1] and convert to 3 channels for CNN compatibility.
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.grayscale_to_rgb(image)
    return image, tf.cast(label, tf.float32)

def get_augmentation_layer():
    # Mild augmentation is used only for training data, never validation/test data.
    return tf.keras.Sequential([
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomContrast(0.1),
    ], name='augmentation')

def dataframe_to_dataset(dataframe, shuffle=False, augment=False, augmentation_layer=None):
    # Convert the split manifest rows into a performant tf.data pipeline.
    paths = dataframe['path'].to_numpy()
    labels = dataframe['class'].map(CLASS_TO_LABEL).to_numpy(dtype='float32')
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(dataframe), seed=RANDOM_SEED, reshuffle_each_iteration=True)

    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)

    if augment and augmentation_layer is not None:
        dataset = dataset.map(
            lambda images, labels: (augmentation_layer(images, training=True), labels),
            num_parallel_calls=AUTOTUNE,
        )

    return dataset.prefetch(AUTOTUNE)

def create_datasets():
    # Build three separate datasets from the reproducible 80/10/10 manifest.
    aug_layer = get_augmentation_layer()
    train_ds = dataframe_to_dataset(split_df[split_df['split'] == 'train'], shuffle=True, augment=True, augmentation_layer=aug_layer)
    val_ds   = dataframe_to_dataset(split_df[split_df['split'] == 'val'])
    test_ds  = dataframe_to_dataset(split_df[split_df['split'] == 'test'])
    class_names = list(EXPECTED_CLASSES)
    return train_ds, val_ds, test_ds, class_names

print('Preprocessing functions ready.')


## 3 - Baseline CNN Model

The baseline model is a small convolutional neural network built from scratch. It uses three `Conv2D + MaxPooling` blocks with 32, 64, and 128 filters, followed by dropout, a dense hidden layer, and a sigmoid output.

The goal is to establish a simple reference model before trying stronger approaches. Because this is a baseline, the architecture is intentionally easy to understand rather than highly optimized.

The next code cell prints both the Keras summary and a compact architecture table with layer names, output shapes, and parameter counts.

The model is compiled with:
- Adam optimizer
- binary cross-entropy loss
- accuracy, precision, and recall metrics

For this medical screening task, recall is especially important because false negatives mean pneumonia cases predicted as normal.


In [ ]:
def build_baseline_model(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)):
    # A small CNN baseline: simple enough to interpret before trying transfer learning.
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        # Convolution blocks learn local visual patterns; pooling reduces spatial size.
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),

        # One sigmoid output is appropriate for binary NORMAL/PNEUMONIA classification.
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model

model = build_baseline_model()
model.summary()

architecture_rows = []
for layer in model.layers:
    output_shape = getattr(layer.output, 'shape', None)
    architecture_rows.append({
        'Layer': layer.__class__.__name__,
        'Name': layer.name,
        'Output shape': str(tuple(output_shape)) if output_shape is not None else 'unknown',
        'Parameters': layer.count_params(),
    })

architecture_df = pd.DataFrame(architecture_rows)
architecture_df.to_csv(RESULTS_DIR / 'baseline_architecture.csv', index=False)
display(architecture_df)


## 4 - Training

Training uses early stopping on `val_loss` with `patience=3` and restores the best validation-loss weights.

`patience=3` means training stops only after validation loss fails to improve for three epochs in a row. This avoids stopping immediately after one noisy validation epoch. With `restore_best_weights=True`, TensorFlow keeps the model weights from the epoch with the best validation loss, not necessarily the final epoch.

The baseline now implements class weighting during training. This does not change the number of training images and does not create offline augmented samples. Instead, it increases the loss contribution of the minority NORMAL class and reduces the loss contribution of the majority PNEUMONIA class. This is a direct implementation response to the class-imbalance feedback while keeping the baseline architecture unchanged.

The baseline training curves saved by the next cell should still be interpreted carefully. Even after replacing the tiny Kaggle validation folder, this is still a single random split of an imbalanced medical image dataset. Validation loss or accuracy can fluctuate because of image difficulty, augmentation effects, class weighting, and the limited baseline architecture.

This validation instability is part of the baseline analysis. It shows that the baseline result should be interpreted cautiously and could be made more reliable with repeated runs, validation-based threshold tuning, or stratified k-fold cross-validation if runtime allows.


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Fix random seeds so the split/shuffling/training run is as reproducible as possible.
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Create the train/validation/test tf.data datasets from the split manifest.
train_ds, val_ds, test_ds, class_names = create_datasets()
print(f'Class index mapping: {class_names}')

# Implement class-imbalance handling for the baseline without changing image counts.
baseline_train_labels = split_df[split_df['split'] == 'train']['class'].map(CLASS_TO_LABEL).to_numpy(dtype=int)
baseline_class_weight_values = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=baseline_train_labels,
)
baseline_class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip([0, 1], baseline_class_weight_values)
}
print('Baseline class weights:', baseline_class_weights)

# Build a fresh baseline model for the single baseline run.
model = build_baseline_model()

# Stop training when validation loss stops improving and keep the best epoch weights.
def get_early_stopping():
    return tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    )


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[get_early_stopping()],
    class_weight=baseline_class_weights,
    verbose=1,
)


In [ ]:
# Save the per-epoch metrics so the report can reference the baseline run.
history_df = pd.DataFrame(history.history)
history_df.to_csv(RESULTS_DIR / 'training_history.csv', index=False)

# Plot the baseline accuracy/loss curves.
# These curves show why a single validation split should be interpreted cautiously.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=150)
plt.show()

# Save the single baseline model separately from the CV fold models.
model.save(MODELS_DIR / 'baseline_cnn.keras')
print(f'Model saved to {MODELS_DIR}/baseline_cnn.keras')


### Baseline Validation Instability

The baseline training curves should be interpreted carefully. Training loss decreases steadily, but validation loss and validation accuracy fluctuate. This can happen because the validation set is still only one random split of an imbalanced medical image dataset, and some X-ray images are harder or noisier than others.

This does not make the baseline invalid, but it means the test metrics should be treated as an initial reference rather than final proof of model reliability. Simpler ways to reduce this issue would be repeated training with different random seeds, class weighting, or threshold tuning. Stratified k-fold cross-validation could also help if more compute is available.


## 5 - Baseline Evaluation and Validation-Based Threshold Diagnostics

This section first reports the baseline CNN on the held-out 10% test split at the default threshold `0.5` as an initial baseline reference. It documents the starting point before validation-based model selection.

To implement the threshold feedback more rigorously, the threshold trade-off analysis is performed on the **validation split**. The selected threshold is then applied to the held-out test split. This avoids choosing a screening threshold directly from the test set.

The notebook reports overall metrics, per-class classification reports, confusion matrices, a ROC curve, a Precision-Recall curve, ROC-AUC, PR-AUC, validation threshold analysis, and selected-threshold test performance.

ROC-AUC is useful because it evaluates how well the model separates the two classes across possible thresholds. Precision-Recall AUC is especially useful for this dataset because the classes are imbalanced and the positive class, pneumonia, is clinically important.

The default threshold of `0.5` is kept as a baseline reference, but it should not automatically be treated as the best medical screening threshold. In screening, false negatives are especially costly, so the notebook selects a validation threshold targeting high pneumonia recall and then reports the resulting test-set trade-off.


In [ ]:
from IPython.display import display
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)


def collect_binary_predictions(dataset, trained_model):
    y_true_local, y_prob_local = [], []
    for images, labels in dataset:
        probs = trained_model.predict(images, verbose=0).flatten()
        y_true_local.extend(labels.numpy().flatten().astype(int))
        y_prob_local.extend(probs)
    return np.array(y_true_local, dtype=int), np.array(y_prob_local, dtype=float)


def make_threshold_table(y_true_local, y_prob_local, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)

    rows = []
    for threshold in thresholds:
        preds_at_threshold = (y_prob_local >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true_local, preds_at_threshold, labels=[0, 1]).ravel()
        precision_value = tp / (tp + fp) if (tp + fp) else 0.0
        recall_value = tp / (tp + fn) if (tp + fn) else 0.0
        specificity_value = tn / (tn + fp) if (tn + fp) else 0.0
        f1_value = 2 * precision_value * recall_value / (precision_value + recall_value) if (precision_value + recall_value) else 0.0
        rows.append({
            'threshold': threshold,
            'precision': precision_value,
            'recall_sensitivity': recall_value,
            'specificity': specificity_value,
            'f1_score': f1_value,
            'false_negatives': int(fn),
            'false_positives': int(fp),
        })
    return pd.DataFrame(rows)


# Keras evaluate uses the compiled metrics at threshold 0.5.
test_results = model.evaluate(test_ds, verbose=1)
metric_names = ['loss', 'accuracy', 'precision', 'recall']
default_test_metrics = dict(zip(metric_names, test_results))

print('Test metrics at default threshold 0.5:')
for k, v in default_test_metrics.items():
    print(f'  {k}: {v:.4f}')

val_y_true, val_y_prob = collect_binary_predictions(val_ds, model)
test_y_true, test_y_prob = collect_binary_predictions(test_ds, model)

expected_test_labels = split_df[split_df['split'] == 'test']['class'].map(CLASS_TO_LABEL).to_numpy(dtype=int)
assert np.array_equal(test_y_true, expected_test_labels), 'test_y_true must match the split manifest labels.'
assert np.all((val_y_prob >= 0.0) & (val_y_prob <= 1.0)), 'Validation probabilities must be between 0 and 1.'
assert np.all((test_y_prob >= 0.0) & (test_y_prob <= 1.0)), 'Test probabilities must be between 0 and 1.'

# Default 0.5 test report remains a baseline reference.
default_threshold = 0.5
test_pred_default = (test_y_prob >= default_threshold).astype(int)
default_report = classification_report(test_y_true, test_pred_default, target_names=class_names, digits=4)
default_report_df = pd.DataFrame(
    classification_report(test_y_true, test_pred_default, target_names=class_names, output_dict=True)
).transpose()
default_report_df.to_csv(RESULTS_DIR / 'classification_report_default_threshold.csv')
with open(RESULTS_DIR / 'classification_report_default_threshold.txt', 'w') as f:
    f.write(default_report)

print('\nTest classification report at default threshold 0.5:')
print(default_report)
display(default_report_df)

cm_default = confusion_matrix(test_y_true, test_pred_default, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm_default, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Baseline CNN, Threshold 0.5')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

# ROC/PR curves on the test split assess ranking quality independently of one fixed threshold.
fpr, tpr, roc_thresholds = roc_curve(test_y_true, test_y_prob)
roc_auc = roc_auc_score(test_y_true, test_y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Baseline CNN (ROC-AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Baseline CNN')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_curve.png', dpi=150)
plt.show()

pr_precision, pr_recall, pr_thresholds = precision_recall_curve(test_y_true, test_y_prob)
pr_auc = average_precision_score(test_y_true, test_y_prob)

plt.figure(figsize=(6, 5))
plt.plot(pr_recall, pr_precision, label=f'Baseline CNN (PR-AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Baseline CNN')
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'precision_recall_curve.png', dpi=150)
plt.show()

# Threshold selection is implemented on validation data, not test data.
TARGET_PNEUMONIA_RECALL = 0.95
threshold_df = make_threshold_table(val_y_true, val_y_prob)
valid_thresholds = threshold_df[threshold_df['recall_sensitivity'] >= TARGET_PNEUMONIA_RECALL]

if not valid_thresholds.empty:
    selected_threshold_row = valid_thresholds.sort_values(
        by=['specificity', 'f1_score', 'threshold'],
        ascending=False,
    ).iloc[0]
    threshold_selection_rule = f'highest validation specificity while recall >= {TARGET_PNEUMONIA_RECALL:.2f}'
else:
    selected_threshold_row = threshold_df.sort_values(
        by=['f1_score', 'recall_sensitivity'],
        ascending=False,
    ).iloc[0]
    threshold_selection_rule = 'best validation F1 because the recall target was not reached'

selected_threshold = float(selected_threshold_row['threshold'])
threshold_df.to_csv(RESULTS_DIR / 'threshold_analysis.csv', index=False)
threshold_df.to_csv(RESULTS_DIR / 'validation_threshold_analysis.csv', index=False)

print('\nValidation threshold trade-off table:')
display(threshold_df)
print(f'Selected threshold: {selected_threshold:.2f} ({threshold_selection_rule})')

# Apply the validation-selected threshold once to the held-out test split.
test_pred_selected = (test_y_prob >= selected_threshold).astype(int)
selected_report = classification_report(test_y_true, test_pred_selected, target_names=class_names, digits=4)
selected_report_df = pd.DataFrame(
    classification_report(test_y_true, test_pred_selected, target_names=class_names, output_dict=True)
).transpose()
selected_report_df.to_csv(RESULTS_DIR / 'classification_report.csv')
selected_report_df.to_csv(RESULTS_DIR / 'classification_report_selected_threshold.csv')
with open(RESULTS_DIR / 'classification_report.txt', 'w') as f:
    f.write(selected_report)
with open(RESULTS_DIR / 'classification_report_selected_threshold.txt', 'w') as f:
    f.write(selected_report)

print(f'\nTest classification report at validation-selected threshold {selected_threshold:.2f}:')
print(selected_report)
display(selected_report_df)

cm_selected = confusion_matrix(test_y_true, test_pred_selected, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm_selected, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - Baseline CNN, Threshold {selected_threshold:.2f}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_selected_threshold.png', dpi=150)
plt.show()

# Save compact metrics for the report.
tn_default, fp_default, fn_default, tp_default = cm_default.ravel()
tn_selected, fp_selected, fn_selected, tp_selected = cm_selected.ravel()
metrics = {
    **{f'default_threshold_{k}': v for k, v in default_test_metrics.items()},
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'selected_threshold': selected_threshold,
    'selected_threshold_precision': tp_selected / (tp_selected + fp_selected) if (tp_selected + fp_selected) else 0.0,
    'selected_threshold_recall': tp_selected / (tp_selected + fn_selected) if (tp_selected + fn_selected) else 0.0,
    'selected_threshold_specificity': tn_selected / (tn_selected + fp_selected) if (tn_selected + fp_selected) else 0.0,
    'default_false_negatives': int(fn_default),
    'default_false_positives': int(fp_default),
    'selected_false_negatives': int(fn_selected),
    'selected_false_positives': int(fp_selected),
}
with open(RESULTS_DIR / 'baseline_metrics.txt', 'w') as f:
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f'{k}: {v:.4f}', file=f)
        else:
            print(f'{k}: {v}', file=f)

print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR-AUC: {pr_auc:.4f}')
print(f'All outputs saved to {OUTPUT_DIR}')


## 6 - Baseline Findings, Limitations, and Next Steps

The baseline section implements the feedback items that benefit from code rather than discussion:
- Image quality is reviewed quantitatively and visually, including a saved grid of the sampled images with the lowest contrast and sharpness proxy.
- Class imbalance is handled in the baseline through class weights during training.
- Threshold choice is implemented using the validation split, then the selected threshold is applied once to the held-out test split.

The generated baseline files provide the current metrics:
- `outputs/results/baseline_metrics.txt` - default-threshold metrics, ROC-AUC, PR-AUC, selected threshold, and false-negative/false-positive counts
- `outputs/results/classification_report_default_threshold.txt` - test report at threshold `0.5`
- `outputs/results/classification_report_selected_threshold.txt` - test report at the validation-selected threshold
- `outputs/results/validation_threshold_analysis.csv` - threshold trade-off table computed on validation predictions
- `outputs/figures/confusion_matrix.png` - default-threshold test confusion matrix
- `outputs/figures/confusion_matrix_selected_threshold.png` - selected-threshold test confusion matrix

The baseline remains a reference model, not the final model. Its validation behavior should be interpreted cautiously because this is a single train/validation/test split of a medical image dataset, and validation behavior can fluctuate because of image difficulty, augmentation effects, class weighting, and the limited baseline architecture.

Important limitations carried forward into final model selection:
- False negatives for pneumonia remain especially important.
- The split is image-level because patient IDs are unavailable, so patient-level leakage cannot be fully ruled out.
- Final architecture decisions are based on validation results, while the test split is reserved for the selected model and selected threshold.


## 7 - Transfer Learning: MobileNetV2

MobileNetV2 is used as an efficient transfer-learning model. The ImageNet-trained base model is frozen and a small binary classification head is trained on the chest X-ray data.

The model includes the correct MobileNetV2 `preprocess_input` transformation inside the model graph. The dataset pipeline provides images scaled to `[0, 1]`, so the model first converts them back to `[0, 255]` before applying the ImageNet preprocessing expected by MobileNetV2.

The same train/validation/test split, training-only augmentation pipeline, class weights, early stopping strategy, and evaluation metrics are used to keep the comparison fair.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights are calculated once from the training labels and reused for transfer learning models.
train_labels = np.concatenate([labels.numpy().astype(int) for _, labels in train_ds])

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels,
)

class_weights = dict(zip(np.unique(train_labels), class_weight_values))
print("Class weights:", class_weights)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model


def build_mobilenet_model(input_shape=(224, 224, 3)):
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
    )
    base_model.trainable = False

    inputs = tf.keras.layers.Input(shape=input_shape)
    x = tf.keras.layers.Lambda(
        lambda images: tf.keras.applications.mobilenet_v2.preprocess_input(images * 255.0),
        name='mobilenetv2_preprocess',
    )(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    transfer_model = Model(inputs=inputs, outputs=outputs, name='MobileNetV2_transfer')

    transfer_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return transfer_model


mobilenet_model = build_mobilenet_model(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
mobilenet_model.summary()


In [ ]:
history_mobilenet = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[get_early_stopping()],
    class_weight=class_weights,
    verbose=1,
)

mobilenet_history_df = pd.DataFrame(history_mobilenet.history)
mobilenet_history_df.to_csv(RESULTS_DIR / 'mobilenet_training_history.csv', index=False)
mobilenet_model.save(MODELS_DIR / 'mobilenetv2_transfer.keras')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mobilenet_history_df[['loss', 'val_loss']].plot(ax=axes[0])
axes[0].set_title('MobileNetV2 Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

mobilenet_history_df[['accuracy', 'val_accuracy']].plot(ax=axes[1])
axes[1].set_title('MobileNetV2 Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mobilenet_training_curves.png', dpi=150)
plt.show()

print(f"Saved MobileNetV2 history to {RESULTS_DIR / 'mobilenet_training_history.csv'}")
print(f"Saved MobileNetV2 model to {MODELS_DIR / 'mobilenetv2_transfer.keras'}")


## 8 - Transfer Learning: DenseNet121

DenseNet121 is used as a second transfer-learning model. It is larger than MobileNetV2 and is included to compare a lightweight transfer-learning approach with a stronger feature-reuse architecture.

As with MobileNetV2, the model includes the correct ImageNet `preprocess_input` transformation inside the model graph while reusing the same split, augmentation, class weights, and early-stopping setup.


In [ ]:
def build_densenet_model(input_shape=(224, 224, 3)):
    base_model = tf.keras.applications.DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
    )
    base_model.trainable = False

    inputs = tf.keras.layers.Input(shape=input_shape)
    x = tf.keras.layers.Lambda(
        lambda images: tf.keras.applications.densenet.preprocess_input(images * 255.0),
        name='densenet121_preprocess',
    )(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    transfer_model = tf.keras.Model(inputs=inputs, outputs=outputs, name='DenseNet121_transfer')

    transfer_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return transfer_model


densenet_model = build_densenet_model(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
densenet_model.summary()


In [ ]:
history_densenet = densenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[get_early_stopping()],
    class_weight=class_weights,
    verbose=1,
)

densenet_history_df = pd.DataFrame(history_densenet.history)
densenet_history_df.to_csv(RESULTS_DIR / 'densenet_training_history.csv', index=False)
densenet_model.save(MODELS_DIR / 'densenet121_transfer.keras')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
densenet_history_df[['loss', 'val_loss']].plot(ax=axes[0])
axes[0].set_title('DenseNet121 Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

densenet_history_df[['accuracy', 'val_accuracy']].plot(ax=axes[1])
axes[1].set_title('DenseNet121 Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'densenet_training_curves.png', dpi=150)
plt.show()

print(f"Saved DenseNet121 history to {RESULTS_DIR / 'densenet_training_history.csv'}")
print(f"Saved DenseNet121 model to {MODELS_DIR / 'densenet121_transfer.keras'}")


## 9 - Evaluation Helper for Fair Model Comparison

The next function evaluates any trained model on a supplied dataset and returns the same metrics for every model. Model comparison is performed on the validation split, not the test split. This keeps architecture decisions separate from the final selected-model test evaluation.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
)

def collect_predictions(trained_model, dataset):
    y_true_local = []
    y_prob_local = []

    for images, labels in dataset:
        probabilities = trained_model.predict(images, verbose=0).flatten()
        y_true_local.extend(labels.numpy().astype(int))
        y_prob_local.extend(probabilities)

    return np.array(y_true_local, dtype=int), np.array(y_prob_local, dtype=float)


def summarize_predictions(model_name, y_true_local, y_prob_local, threshold=0.5, split_name='validation'):
    y_pred_local = (y_prob_local >= threshold).astype(int)
    cm_local = confusion_matrix(y_true_local, y_pred_local, labels=[0, 1])
    tn, fp, fn, tp = cm_local.ravel()

    result = {
        'Model': model_name,
        'Split': split_name,
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_true_local, y_pred_local),
        'Precision': precision_score(y_true_local, y_pred_local, zero_division=0),
        'Recall': recall_score(y_true_local, y_pred_local, zero_division=0),
        'F1-score': f1_score(y_true_local, y_pred_local, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true_local, y_prob_local),
        'PR-AUC': average_precision_score(y_true_local, y_prob_local),
        'False Negatives': int(fn),
        'False Positives': int(fp),
        'Specificity': tn / (tn + fp) if (tn + fp) else 0.0,
    }

    print(f'\n{model_name} classification report on {split_name}, threshold={threshold}')
    print(classification_report(
        y_true_local,
        y_pred_local,
        target_names=class_names,
        digits=4,
        zero_division=0,
    ))
    print(f'{model_name} confusion matrix on {split_name}:')
    print(cm_local)

    return result, cm_local, y_pred_local


def evaluate_binary_model(model_name, trained_model, dataset, threshold=0.5, split_name='validation'):
    y_true_local, y_prob_local = collect_predictions(trained_model, dataset)
    result, cm_local, y_pred_local = summarize_predictions(
        model_name,
        y_true_local,
        y_prob_local,
        threshold=threshold,
        split_name=split_name,
    )
    return result, cm_local, y_true_local, y_prob_local, y_pred_local


def threshold_tradeoff_table(y_true_local, y_prob_local, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)

    rows = []
    for threshold in thresholds:
        preds_at_threshold = (y_prob_local >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true_local, preds_at_threshold, labels=[0, 1]).ravel()
        precision_value = tp / (tp + fp) if (tp + fp) else 0.0
        recall_value = tp / (tp + fn) if (tp + fn) else 0.0
        specificity_value = tn / (tn + fp) if (tn + fp) else 0.0
        f1_value = 2 * precision_value * recall_value / (precision_value + recall_value) if (precision_value + recall_value) else 0.0
        rows.append({
            'threshold': threshold,
            'precision': precision_value,
            'recall_sensitivity': recall_value,
            'specificity': specificity_value,
            'f1_score': f1_value,
            'false_negatives': int(fn),
            'false_positives': int(fp),
        })

    return pd.DataFrame(rows)


In [ ]:
baseline_val_result, baseline_val_cm, baseline_val_y_true, baseline_val_y_prob, baseline_val_y_pred = evaluate_binary_model(
    'Baseline CNN',
    model,
    val_ds,
    split_name='validation',
)

mobilenet_val_result, mobilenet_val_cm, mobilenet_val_y_true, mobilenet_val_y_prob, mobilenet_val_y_pred = evaluate_binary_model(
    'MobileNetV2',
    mobilenet_model,
    val_ds,
    split_name='validation',
)

densenet_val_result, densenet_val_cm, densenet_val_y_true, densenet_val_y_prob, densenet_val_y_pred = evaluate_binary_model(
    'DenseNet121',
    densenet_model,
    val_ds,
    split_name='validation',
)


## 10 - Validation-Based Model Comparison and Threshold Selection

The table below compares models on the validation split. This is the split used for model selection and architecture decisions. The test split is not used for this comparison.

Because this is a medical screening-style task, the threshold is also selected on validation data. The notebook targets high pneumonia recall first, then chooses the threshold with the best specificity among thresholds that still meet the recall target. This makes the false-negative trade-off explicit instead of assuming that `0.5` is automatically appropriate.


In [ ]:
validation_comparison_df = pd.DataFrame([
    baseline_val_result,
    mobilenet_val_result,
    densenet_val_result,
])
validation_comparison_df = validation_comparison_df.sort_values(
    by=['PR-AUC', 'ROC-AUC', 'Recall'],
    ascending=False,
).reset_index(drop=True)
validation_comparison_df.to_csv(RESULTS_DIR / 'validation_model_comparison.csv', index=False)

comparison_df = validation_comparison_df.copy()
display(validation_comparison_df)
print(f"Saved validation model comparison to {RESULTS_DIR / 'validation_model_comparison.csv'}")

model_lookup = {
    'Baseline CNN': model,
    'MobileNetV2': mobilenet_model,
    'DenseNet121': densenet_model,
}
validation_prediction_lookup = {
    'Baseline CNN': (baseline_val_y_true, baseline_val_y_prob),
    'MobileNetV2': (mobilenet_val_y_true, mobilenet_val_y_prob),
    'DenseNet121': (densenet_val_y_true, densenet_val_y_prob),
}

SELECTED_MODEL_NAME = validation_comparison_df.iloc[0]['Model']
selected_model = model_lookup[SELECTED_MODEL_NAME]
selected_val_y_true, selected_val_y_prob = validation_prediction_lookup[SELECTED_MODEL_NAME]

TARGET_PNEUMONIA_RECALL = 0.95
validation_threshold_df = threshold_tradeoff_table(selected_val_y_true, selected_val_y_prob)
valid_thresholds = validation_threshold_df[
    validation_threshold_df['recall_sensitivity'] >= TARGET_PNEUMONIA_RECALL
]

if not valid_thresholds.empty:
    selected_threshold_row = valid_thresholds.sort_values(
        by=['specificity', 'f1_score', 'threshold'],
        ascending=False,
    ).iloc[0]
    threshold_selection_rule = f'highest specificity while recall >= {TARGET_PNEUMONIA_RECALL:.2f}'
else:
    selected_threshold_row = validation_threshold_df.sort_values(
        by=['f1_score', 'recall_sensitivity'],
        ascending=False,
    ).iloc[0]
    threshold_selection_rule = 'best validation F1 because the recall target was not reached'

SELECTED_THRESHOLD = float(selected_threshold_row['threshold'])
validation_threshold_df.to_csv(RESULTS_DIR / 'validation_threshold_analysis.csv', index=False)

selection_summary = {
    'selected_model': SELECTED_MODEL_NAME,
    'selection_metric': 'validation PR-AUC, then ROC-AUC and recall',
    'target_pneumonia_recall': TARGET_PNEUMONIA_RECALL,
    'selected_threshold': SELECTED_THRESHOLD,
    'threshold_selection_rule': threshold_selection_rule,
}
pd.DataFrame([selection_summary]).to_csv(RESULTS_DIR / 'model_selection_summary.csv', index=False)

print(f'Selected model from validation PR-AUC: {SELECTED_MODEL_NAME}')
print(f'Selected threshold: {SELECTED_THRESHOLD:.2f} ({threshold_selection_rule})')
print(f"Saved validation threshold analysis to {RESULTS_DIR / 'validation_threshold_analysis.csv'}")
display(validation_threshold_df)


## 11 - Selected-Model Test Evaluation

After selecting the model and threshold on validation data, the selected configuration is evaluated once on the held-out test split. This keeps the test set closer to its intended role: estimating performance after model and threshold choices have already been made.


In [ ]:
selected_test_y_true, selected_test_y_prob = collect_predictions(selected_model, test_ds)
selected_test_result, selected_test_cm, selected_test_y_pred = summarize_predictions(
    SELECTED_MODEL_NAME,
    selected_test_y_true,
    selected_test_y_prob,
    threshold=SELECTED_THRESHOLD,
    split_name='test',
)

selected_test_df = pd.DataFrame([selected_test_result])
selected_test_df.to_csv(RESULTS_DIR / 'selected_model_test_metrics.csv', index=False)

plt.figure(figsize=(6, 5))
sns.heatmap(
    selected_test_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'{SELECTED_MODEL_NAME} Test Confusion Matrix, Threshold {SELECTED_THRESHOLD:.2f}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'selected_model_test_confusion_matrix.png', dpi=150)
plt.show()

display(selected_test_df)
print(f"Saved selected-model test metrics to {RESULTS_DIR / 'selected_model_test_metrics.csv'}")


In [ ]:
gradcam_model_name = SELECTED_MODEL_NAME if 'SELECTED_MODEL_NAME' in globals() else 'Baseline CNN'
gradcam_model = selected_model if 'selected_model' in globals() else model


def find_last_conv_target(trained_model):
    for layer in reversed(trained_model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return {'kind': 'top_level', 'layer_name': layer.name}
        if isinstance(layer, tf.keras.Model):
            for nested_layer in reversed(layer.layers):
                if isinstance(nested_layer, tf.keras.layers.Conv2D):
                    return {
                        'kind': 'nested_model',
                        'parent_name': layer.name,
                        'layer_name': nested_layer.name,
                    }
    raise ValueError(f'No Conv2D layer found for Grad-CAM in {trained_model.name}.')


gradcam_target = find_last_conv_target(gradcam_model)
best_model_name = gradcam_model_name
best_model = gradcam_model

print('Model:', best_model_name)
print('Grad-CAM target:', gradcam_target)


## 12 - Grad-CAM Explainability

Grad-CAM visualizes which image regions influenced a CNN prediction. The goal is to check whether the selected model appears to rely on plausible image regions, such as the lung fields, rather than obviously irrelevant corners, labels, or borders.

The implementation supports both the custom baseline CNN and nested transfer-learning models such as MobileNetV2 and DenseNet121. For transfer-learning models, the Grad-CAM target is the last convolution layer inside the frozen pretrained base.

Grad-CAM is an interpretability support tool. It does not prove clinical correctness, and the heatmaps should ideally be reviewed by medical experts. The notebook saves both the figures and a small CSV summary containing true labels, predicted labels, probabilities, and Grad-CAM filenames.


In [ ]:
def build_gradcam_model(trained_model, target):
    inputs = trained_model.inputs
    x = inputs[0]
    conv_outputs = None

    for layer in trained_model.layers:
        if isinstance(layer, tf.keras.layers.InputLayer):
            continue

        if target['kind'] == 'nested_model' and layer.name == target['parent_name']:
            nested_model = layer
            nested_conv_layer = nested_model.get_layer(target['layer_name'])
            nested_extractor = tf.keras.Model(
                nested_model.inputs,
                [nested_conv_layer.output, nested_model.output],
            )
            conv_outputs, x = nested_extractor(x)
        else:
            x = layer(x)
            if target['kind'] == 'top_level' and layer.name == target['layer_name']:
                conv_outputs = x

    if conv_outputs is None:
        raise ValueError('Could not connect Grad-CAM convolution target.')

    return tf.keras.Model(inputs=inputs, outputs=[conv_outputs, x])


def make_gradcam_heatmap(img_array, trained_model, target):
    grad_model = build_gradcam_model(trained_model, target)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def show_gradcam(image, true_label, filename, threshold=None):
    threshold = SELECTED_THRESHOLD if threshold is None and 'SELECTED_THRESHOLD' in globals() else 0.5
    img_array = tf.expand_dims(image, axis=0)
    probability = float(best_model.predict(img_array, verbose=0).flatten()[0])
    predicted_label = int(probability >= threshold)
    heatmap = make_gradcam_heatmap(img_array, best_model, gradcam_target)
    heatmap = np.uint8(255 * heatmap)
    heatmap = Image.fromarray(heatmap).resize((image.shape[1], image.shape[0]))
    heatmap = np.asarray(heatmap) / 255.0

    image_np = image.numpy()
    grayscale = image_np[:, :, 0]

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(grayscale, cmap='gray')
    plt.title(f"True: {class_names[int(true_label)]}\nPred: {class_names[predicted_label]} ({probability:.3f})")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(grayscale, cmap='gray')
    plt.imshow(heatmap, cmap='jet', alpha=0.35)
    plt.title('Grad-CAM overlay')
    plt.axis('off')

    plt.tight_layout()
    output_path = FIGURES_DIR / filename
    plt.savefig(output_path, dpi=150)
    plt.show()
    print(f'Saved Grad-CAM figure to {output_path}')

    return {
        'model': best_model_name,
        'gradcam_target': str(gradcam_target),
        'true_label': class_names[int(true_label)],
        'predicted_label': class_names[predicted_label],
        'pneumonia_probability': probability,
        'threshold': threshold,
        'figure': str(output_path),
        'interpretation_note': 'Review whether highlighted regions overlap lung fields; Grad-CAM is supportive, not diagnostic evidence.',
    }


In [ ]:
# Show and save three Grad-CAM examples from the test set.
example_count = 0
gradcam_rows = []

for batch_images, batch_labels in test_ds:
    for image, label in zip(batch_images, batch_labels):
        example_count += 1
        row = show_gradcam(
            image=image,
            true_label=label.numpy(),
            filename=f"gradcam_{best_model_name.lower().replace(' ', '_')}_{example_count}.png",
            threshold=SELECTED_THRESHOLD if 'SELECTED_THRESHOLD' in globals() else 0.5,
        )
        gradcam_rows.append(row)

        if example_count >= 3:
            break

    if example_count >= 3:
        break

gradcam_summary_df = pd.DataFrame(gradcam_rows)
gradcam_summary_df.to_csv(RESULTS_DIR / 'gradcam_summary.csv', index=False)
display(gradcam_summary_df)
print(f"Saved Grad-CAM summary to {RESULTS_DIR / 'gradcam_summary.csv'}")


## Optional: Download Outputs from Colab

Run the next cell at the end of a Colab execution to zip and download the complete `outputs/` folder. Locally, the cell simply prints the folder path because the files are already in the project directory.


In [ ]:
import shutil

if 'google.colab' in sys.modules:
    from google.colab import files

    zip_base = str(OUTPUT_DIR)
    zip_path = shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
    print(f'Created archive: {zip_path}')
    files.download(zip_path)
else:
    print(f'Outputs are available locally at: {OUTPUT_DIR.resolve()}')


## Conclusion

This project builds a reproducible pipeline for binary chest X-ray classification into NORMAL and PNEUMONIA. The workflow includes data inspection, stratified splitting, online augmentation, class weighting, a custom CNN baseline, two transfer-learning models, validation-based model selection, validation-based threshold selection, final test evaluation, and Grad-CAM interpretation.

The strongest methodological choice is that architecture and threshold decisions are made on the validation split. The test split is used only after those choices are fixed. This avoids tuning directly on the test data and makes the final evaluation more meaningful.

The project still has important limitations. The split is image-level because patient identifiers are unavailable, so patient-level leakage cannot be fully ruled out. The dataset may not represent all hospitals, devices, or patient populations. Grad-CAM can help inspect model attention, but it does not prove medical correctness. A clinically useful system would need external validation, subgroup analysis, calibration, patient-level splitting, and review by medical experts.
